In [1]:
# %% ===================== SHARED: imports + layers (run once) =====================
import tensorflow as tf
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, MaxPooling2D, Dense, Dropout,
                                     GlobalAveragePooling2D, Reshape, Multiply)
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam

input_shape = (224, 224, 3)
num_classes = 5
lr = 1e-5
bs = 32
ep = 30

class ASCFBlock(tf.keras.layers.Layer):
    def __init__(self, num_clusters, **kwargs):
        super(ASCFBlock, self).__init__(**kwargs)
        self.num_clusters = num_clusters
        self.conv1x1 = tf.keras.layers.Conv2D(1, kernel_size=1, use_bias=False)
        self.dense = tf.keras.layers.Dense(num_clusters, activation='sigmoid')

    def call(self, feature_maps):
        batch_size = tf.shape(feature_maps)[0]
        h, w, num_channels = feature_maps.shape[1:]
        maps_per_cluster = num_channels // self.num_clusters

        reshaped = tf.reshape(feature_maps, (batch_size, h, w, self.num_clusters, maps_per_cluster))
        temp_batch = batch_size * self.num_clusters
        conv_input = tf.reshape(reshaped, (temp_batch, h, w, maps_per_cluster))

        fused = self.conv1x1(conv_input)
        fused = tf.reshape(fused, (batch_size, h, w, self.num_clusters))

        squeezed = tf.reduce_mean(fused, axis=[1, 2])
        weights = self.dense(squeezed)
        weighted = fused * tf.reshape(weights, (batch_size, 1, 1, self.num_clusters))

        final_map = tf.reduce_max(weighted, axis=-1, keepdims=True)
        vector_output = tf.reduce_mean(weighted, axis=[1, 2])

        return final_map, vector_output

    def compute_output_spec(self, input_spec):
        batch_size, h, w, num_channels = input_spec.shape
        if num_channels % self.num_clusters != 0:
            raise ValueError(f"Number of channels ({num_channels}) must be divisible by num_clusters ({self.num_clusters})")

        final_map_spec = tf.keras.KerasTensor(shape=(batch_size, h, w, 1), dtype=tf.float32)
        vector_output_spec = tf.keras.KerasTensor(shape=(batch_size, self.num_clusters), dtype=tf.float32)
        return final_map_spec, vector_output_spec

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"num_clusters": self.num_clusters})
        return cfg


def se_block(x, ratio=16):
    channels = x.shape[-1]
    se = GlobalAveragePooling2D()(x)
    se = Dense(channels // ratio, activation='relu')(se)
    se = Dense(channels, activation='sigmoid')(se)
    se = Reshape((1, 1, channels))(se)
    return Multiply()([x, se])


# input_shape, num_classes, lr must be defined before running a model block


# %% ===================== VARIANT 1: Backbone + GAP -> Dense(1024)  (no SE) =====================
input_layer = Input(shape=input_shape)
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
x = base_model(input_layer)

x = Conv2D(1024, (3, 3), padding='same', activation='relu')(x)
x = BatchNormalization()(x)
x = MaxPooling2D((1, 1))(x)

x = Conv2D(2048, (3, 3), padding='same', activation='relu')(x)
x = BatchNormalization()(x)
# x = se_block(x)                      # removed: no SE in this variant
x = MaxPooling2D((1, 1))(x)

x = GlobalAveragePooling2D()(x)        # bottleneck = 2048

x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)

output_layer = Dense(num_classes, activation='softmax')(x)
model_v1 = Model(inputs=input_layer, outputs=output_layer)
model_v1.compile(optimizer=Adam(learning_rate=lr), loss='categorical_crossentropy', metrics=['accuracy'])
model_v1.summary()
print("V1 params (M):", model_v1.count_params() / 1e6)


# %% ===================== VARIANT 2: Backbone + GAP -> Dense(16) -> Dense(1024)  (no SE) =====================
input_layer = Input(shape=input_shape)
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
x = base_model(input_layer)

x = Conv2D(1024, (3, 3), padding='same', activation='relu')(x)
x = BatchNormalization()(x)
x = MaxPooling2D((1, 1))(x)

x = Conv2D(2048, (3, 3), padding='same', activation='relu')(x)
x = BatchNormalization()(x)
# x = se_block(x)                      # removed: no SE in this variant
x = MaxPooling2D((1, 1))(x)

x = GlobalAveragePooling2D()(x)
x = Dense(16, activation='relu')(x)    # bottleneck = 16

x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)

output_layer = Dense(num_classes, activation='softmax')(x)
model_v2 = Model(inputs=input_layer, outputs=output_layer)
model_v2.compile(optimizer=Adam(learning_rate=lr), loss='categorical_crossentropy', metrics=['accuracy'])
model_v2.summary()
print("V2 params (M):", model_v2.count_params() / 1e6)


# %% ===================== VARIANT 3: Backbone + SE + GAP  (your model_gap, unchanged) =====================
input_layer = Input(shape=input_shape)
base_model_gap = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
x = base_model_gap(input_layer)

x = Conv2D(1024, (3, 3), padding='same', activation='relu')(x)
x = BatchNormalization()(x)
x = MaxPooling2D((1, 1))(x)

x = Conv2D(2048, (3, 3), padding='same', activation='relu')(x)
x = BatchNormalization()(x)
x = se_block(x)
x = MaxPooling2D((1, 1))(x)

x = GlobalAveragePooling2D()(x)        # bottleneck = 2048

x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)

output_layer = Dense(num_classes, activation='softmax')(x)
model_v3 = Model(inputs=input_layer, outputs=output_layer)
model_v3.compile(optimizer=Adam(learning_rate=lr), loss='categorical_crossentropy', metrics=['accuracy'])
model_v3.summary()
print("V3 params (M):", model_v3.count_params() / 1e6)


# %% ===================== VARIANT 4: Backbone + CABlock (no SE) =====================
input_layer = Input(shape=input_shape)
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
x = base_model(input_layer)

x = Conv2D(1024, (3, 3), padding='same', activation='relu')(x)
x = BatchNormalization()(x)
x = MaxPooling2D((1, 1))(x)

x = Conv2D(2048, (3, 3), padding='same', activation='relu')(x)
x = BatchNormalization()(x)
# x = se_block(x)                      # removed: no SE in this variant
x = MaxPooling2D((1, 1))(x)

ascf_layer = ASCFBlock(num_clusters=16)
_, x = ascf_layer(x)                   # bottleneck = 16

x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)

output_layer = Dense(num_classes, activation='softmax')(x)
model_v4 = Model(inputs=input_layer, outputs=output_layer)
model_v4.compile(optimizer=Adam(learning_rate=lr), loss='categorical_crossentropy', metrics=['accuracy'])
model_v4.summary()
print("V4 params (M):", model_v4.count_params() / 1e6)


# %% ===================== VARIANT 5: Backbone + SE + CABlock (Proposed) =====================
input_layer = Input(shape=input_shape)
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
x = base_model(input_layer)

x = Conv2D(1024, (3, 3), padding='same', activation='relu')(x)
x = BatchNormalization()(x)
x = MaxPooling2D((1, 1))(x)

x = Conv2D(2048, (3, 3), padding='same', activation='relu')(x)
x = BatchNormalization()(x)
x = se_block(x)
x = MaxPooling2D((1, 1))(x)

ascf_layer = ASCFBlock(num_clusters=16)
_, x = ascf_layer(x)                   # bottleneck = 16

x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)

output_layer = Dense(num_classes, activation='softmax')(x)
model_v5 = Model(inputs=input_layer, outputs=output_layer)
model_v5.compile(optimizer=Adam(learning_rate=lr), loss='categorical_crossentropy', metrics=['accuracy'])
model_v5.summary()
print("V5 params (M):", model_v5.count_params() / 1e6)

I0000 00:00:1794494685.104925      49 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794494707.266921      49 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1794494712.719316      49 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 7, 7, 1024)     │    18,875,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 7, 7, 1024)     │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 7, 7, 1024)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 7, 7, 2048)     │    18,876,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 7, 7, 2048)     │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 2048)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1024)           │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │         2,565 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 63,977,349 (244.05 MB)

 Trainable params: 63,918,085 (243.83 MB)

... (22 outputs left out)

 Total params: 62,423,445 (238.13 MB)

 Trainable params: 62,364,181 (237.90 MB)

 Non-trainable params: 59,264 (231.50 KB)

V5 params (M): 62.423445


In [2]:
import numpy as np
import pandas as pd

def count(weights):
    return int(sum(np.prod(w.shape) for w in weights))

variants = {
    "V1: Backbone + GAP -> Dense(1024)":        model_v1,
    "V2: Backbone + GAP -> Dense(16)":          model_v2,
    "V3: Backbone + SE + GAP":                  model_v3,
    "V4: Backbone + CABlock":                   model_v4,
    "V5: Backbone + SE + CABlock (Proposed)":   model_v5,
}

rows = []
for name, m in variants.items():
    trainable = count(m.trainable_weights)
    non_trainable = count(m.non_trainable_weights)
    total = m.count_params()
    assert total == trainable + non_trainable, f"Mismatch in {name}"
    rows.append({
        "Model": name,
        "Total": total,
        "Trainable": trainable,
        "Non-trainable": non_trainable,
        "Total (M)": round(total / 1e6, 2),
        "Trainable (M)": round(trainable / 1e6, 2),
        "Non-trainable (M)": round(non_trainable / 1e6, 4),
    })

param_df = pd.DataFrame(rows)

param_df["Total vs V5 (M)"] = (param_df["Total"] - param_df["Total"].iloc[-1]) / 1e6
param_df["Total vs V5 (M)"] = param_df["Total vs V5 (M)"].round(2)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}".rstrip("0").rstrip("."))
print(param_df.to_string(index=False, formatters={
    "Total": "{:,}".format, "Trainable": "{:,}".format, "Non-trainable": "{:,}".format}))

param_df.to_csv("model_parameters.csv", index=False)
param_df

                                 Model      Total  Trainable Non-trainable  Total (M)  Trainable (M)  Non-trainable (M)  Total vs V5 (M)
     V1: Backbone + GAP -> Dense(1024) 63,977,349 63,918,085        59,264      63.98          63.92             0.0593             1.55
       V2: Backbone + GAP -> Dense(16) 61,929,365 61,870,101        59,264      61.93          61.87             0.0593            -0.49
               V3: Backbone + SE + GAP 64,503,813 64,444,549        59,264       64.5          64.44             0.0593             2.08
                V4: Backbone + CABlock 61,896,981 61,837,717        59,264       61.9          61.84             0.0593            -0.53
V5: Backbone + SE + CABlock (Proposed) 62,423,445 62,364,181        59,264      62.42          62.36             0.0593                0


,Model,Total,Trainable,Non-trainable,Total (M),Trainable (M),Non-trainable (M),Total vs V5 (M)
0,V1: Backbone + GAP -> Dense(1024),63977349,63918085,59264,63.98,63.92,0.0593,1.55
1,V2: Backbone + GAP -> Dense(16),61929365,61870101,59264,61.93,61.87,0.0593,-0.49
2,V3: Backbone + SE + GAP,64503813,64444549,59264,64.5,64.44,0.0593,2.08
3,V4: Backbone + CABlock,61896981,61837717,59264,61.9,61.84,0.0593,-0.53
4,V5: Backbone + SE + CABlock (Proposed),62423445,62364181,59264,62.42,62.36,0.0593,0
